In [57]:
import torch
import torch.nn.functional as F 
import matplotlib.pyplot as plt

In [58]:
words = open("names.txt", 'r').read().splitlines()

In [59]:
# build a vocabulary
chars = sorted(list(set(''.join(words))))
stoi = { ele:idx+1 for idx, ele in enumerate(chars)}
stoi['.'] = 0
itos = { idx: ele for ele, idx in stoi.items()}


In [64]:
# build a dataset
block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    # print(w)
    
    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(f"{"".join( itos[c] for c in context)} ----> {ch}")
        context = context[1:] + [ix]
        
X = torch.tensor(X)
Y = torch.tensor(Y)

In [65]:
C = torch.randn((27,2)) # These are for embeddings

W1 = torch.randn((6, 100)) # These are weights for layer 1 
b1 = torch.randn(100) # Bias for layer 1

W2 = torch.rand((100, 27)) # These are weights for layer 2 
b2 = torch.rand(27) # Bias for layer 2

parametrs = [ C, W1, b1, W2, b2]

# These are the paramets

In [66]:
for p in parametrs:
    p.requires_grad = True

In [ ]:
embeddings = C[X]
h= torch.tanh(embeddings.view(-1, 6) @ W1 + b1) # Forward pass for Layer 1 tanh(inputs * W1 + b1)

logits = h @ W2 + b2 # Forward pass for Layer 2 
# This is called Classification

counts = logits.exp() # Getting outputs
probs = counts / counts.sum(1, keepdim=True) # Convert into probablities
loss = -probs[torch.arange(32), Y].log().mean() # Calculate Log likely hood

loss_by_cross_entropy = F.cross_entropy(logits, Y) # This is same as above 

In [ ]:
for _ in range(100):
    # use batches
    batch =  torch.randint(0, X.shape[0], (32,))
    
    # Forward Pass
    embeddings = C[X[batch]]
    h= torch.tanh(embeddings.view(-1, 6) @ W1 + b1) # Forward pass for Layer 1 tanh(inputs * W1 + b1)
    logits = h @ W2 + b2 # Forward pass for Layer 2 
    loss = F.cross_entropy(logits, Y[batch]) # This is same as above

    # Set parametrs to None
    for p in parametrs:
        p.grad = None
    
    # Backward Pass    
    loss.backward()

    # Update parameters 
    for p in parametrs:
        p.data += (-0.1 * p.grad) # type: ignore
    
print(loss.item())

2.605764627456665


In [78]:
# Rather than doing forward pass and backward pass completly on the entire set we focus on one single batch
batch =  torch.randint(0, X.shape[0], (32,))

In [79]:
batch

tensor([ 85854, 142467, 122913, 160128, 148905,  65806, 104541, 176661, 206953,
        209665,  44460,  51558, 148408, 159327, 191522,  92214, 112454,  85849,
          4004,  25891, 143036, 116433, 204735,  82474, 179497, 163639,   5051,
        176490, 193347, 203959, 177002,  95685])